# Dataset sintético de ECG para Edge Impulse

Este notebook genera **200 señales sintéticas por clase** (600 en total) con NeuroKit2. Cada señal dura 20 segundos y se simula con el método ECGSYN. Las clases se distinguen por su frecuencia cardíaca objetivo:

- **Bradicardia:** 40–55 BPM.
- **Ritmo sinusal normal:** 60–100 BPM.
- **Taquicardia:** 105–150 BPM.

Los archivos se guardan bajo `dataset/<clase>/raw/` y `dataset/<clase>/formateado/`. Los CSV de `formateado` usan las columnas `timestamp,ecg`, con marcas de tiempo en milisegundos y el nombre de la clase como prefijo para facilitar la importación a Edge Impulse. Son datos sintéticos para pruebas y clasificación; no son registros clínicos.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np
import pandas as pd

NOTEBOOK_FOLDER_NAME = "7_3_neurokit_edgeimpulse"
current_folder = Path.cwd()
if current_folder.name == NOTEBOOK_FOLDER_NAME:
    NOTEBOOK_DIR = current_folder
else:
    NOTEBOOK_DIR = current_folder / "7_edge_impulse" / NOTEBOOK_FOLDER_NAME
if not NOTEBOOK_DIR.is_dir():
    raise FileNotFoundError(
        f"No se encontró la carpeta del notebook desde {current_folder}. "
        "Abre el notebook desde la raíz del repositorio o desde su propia carpeta."
    )

DATASET_DIR = NOTEBOOK_DIR / "dataset"
NUM_SIGNALS_PER_CLASS = 200
DURATION_SECONDS = 20
SAMPLING_RATE_HZ = 250
NUM_SAMPLES = DURATION_SECONDS * SAMPLING_RATE_HZ
RANDOM_SEED = 2026

CLASS_CONFIG = {
    "bradicardia": {"display_name": "Bradicardia", "heart_rate_range": (40, 55)},
    "ritmo_sinusal_normal": {
        "display_name": "Ritmo sinusal normal",
        "heart_rate_range": (60, 100),
    },
    "taquicardia": {"display_name": "Taquicardia", "heart_rate_range": (105, 150)},
}

print(f"Carpeta de salida: {DATASET_DIR}")
print(f"Señales por clase: {NUM_SIGNALS_PER_CLASS}; duración: {DURATION_SECONDS} s; muestreo: {SAMPLING_RATE_HZ} Hz")

## Formato y organización

Para cada clase se crean `raw/` y `formateado/`. El CSV crudo conserva el tiempo en segundos; el CSV formateado usa el esquema de series temporales de Edge Impulse (`timestamp` en milisegundos y una columna `ecg`). Cada archivo formateado comienza con el nombre de su clase, por ejemplo `taquicardia.ecg_0001.csv`. El archivo `manifest.csv` registra la clase, la frecuencia cardíaca objetivo y las rutas de ambos CSV.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
manifest_rows = []
time_seconds = np.arange(NUM_SAMPLES, dtype=np.float64) / SAMPLING_RATE_HZ
timestamps_ms = time_seconds * 1000.0

for class_name, class_info in CLASS_CONFIG.items():
    raw_dir = DATASET_DIR / class_name / "raw"
    formatted_dir = DATASET_DIR / class_name / "formateado"
    raw_dir.mkdir(parents=True, exist_ok=True)
    formatted_dir.mkdir(parents=True, exist_ok=True)

    heart_rate_min, heart_rate_max = class_info["heart_rate_range"]
    for signal_index in range(1, NUM_SIGNALS_PER_CLASS + 1):
        target_heart_rate = int(rng.integers(heart_rate_min, heart_rate_max + 1))
        signal_seed = int(rng.integers(0, np.iinfo(np.uint32).max))
        ecg = np.asarray(
            nk.ecg_simulate(
                duration=DURATION_SECONDS,
                sampling_rate=SAMPLING_RATE_HZ,
                heart_rate=target_heart_rate,
                heart_rate_std=0.5,
                method="ecgsyn",
                noise=0.01,
                random_state=signal_seed,
            ),
            dtype=np.float64,
        ).reshape(-1)

        if ecg.size != NUM_SAMPLES or not np.isfinite(ecg).all():
            raise ValueError(
                f"Señal inválida para {class_name}, muestra {signal_index}: "
                f"se esperaban {NUM_SAMPLES} muestras finitas; se obtuvieron {ecg.size}."
            )

        base_filename = f"ecg_{signal_index:04d}.csv"
        raw_path = raw_dir / base_filename
        formatted_path = formatted_dir / f"{class_name}.{base_filename}"

        np.savetxt(
            raw_path,
            np.column_stack((time_seconds, ecg)),
            delimiter=",",
            header="time_s,ecg",
            comments="",
            fmt=("%.6f", "%.8f"),
        )
        np.savetxt(
            formatted_path,
            np.column_stack((timestamps_ms, ecg)),
            delimiter=",",
            header="timestamp,ecg",
            comments="",
            fmt=("%.3f", "%.8f"),
        )

        manifest_rows.append(
            {
                "class_label": class_name,
                "display_name": class_info["display_name"],
                "target_heart_rate_bpm": target_heart_rate,
                "sampling_rate_hz": SAMPLING_RATE_HZ,
                "duration_seconds": DURATION_SECONDS,
                "samples": NUM_SAMPLES,
                "seed": signal_seed,
                "raw_file": raw_path.relative_to(DATASET_DIR).as_posix(),
                "edge_impulse_file": formatted_path.relative_to(DATASET_DIR).as_posix(),
            }
        )

    print(f"{class_info['display_name']}: {NUM_SIGNALS_PER_CLASS} señales generadas.")

manifest_path = DATASET_DIR / "manifest.csv"
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
print(f"\nTotal: {len(manifest_rows)} señales. Manifiesto: {manifest_path}")

## Verificar el dataset

La celda siguiente comprueba que cada clase tenga 200 archivos crudos y 200 archivos formateados, que los CSV tengan los encabezados esperados y que las marcas de tiempo estén espaciadas según la frecuencia de muestreo. Después grafica una señal de ejemplo por clase.

In [ ]:
verification_rows = []
fig, axes = plt.subplots(len(CLASS_CONFIG), 1, figsize=(12, 8), sharex=True)

for axis, (class_name, class_info) in zip(axes, CLASS_CONFIG.items()):
    raw_files = sorted((DATASET_DIR / class_name / "raw").glob("ecg_*.csv"))
    formatted_files = sorted((DATASET_DIR / class_name / "formateado").glob(f"{class_name}.ecg_*.csv"))
    if len(raw_files) != NUM_SIGNALS_PER_CLASS or len(formatted_files) != NUM_SIGNALS_PER_CLASS:
        raise RuntimeError(
            f"{class_name}: se esperaban {NUM_SIGNALS_PER_CLASS} archivos en cada carpeta; "
            f"se encontraron {len(raw_files)} crudos y {len(formatted_files)} formateados."
        )

    raw_header = raw_files[0].read_text(encoding="utf-8").splitlines()[0].split(",")
    formatted_header = formatted_files[0].read_text(encoding="utf-8").splitlines()[0].split(",")
    if raw_header != ["time_s", "ecg"] or formatted_header != ["timestamp", "ecg"]:
        raise ValueError(f"Encabezados inesperados para la clase {class_name}.")

    sample = np.loadtxt(formatted_files[0], delimiter=",", skiprows=1)
    if sample.shape != (NUM_SAMPLES, 2):
        raise ValueError(f"Dimensiones inesperadas en {formatted_files[0]}: {sample.shape}.")
    np.testing.assert_allclose(np.diff(sample[:, 0]), 1000.0 / SAMPLING_RATE_HZ)

    axis.plot(time_seconds, sample[:, 1], linewidth=0.7)
    axis.set_title(class_info["display_name"])
    axis.set_ylabel("Amplitud ECG")
    axis.grid(True, alpha=0.3)
    verification_rows.append(
        {
            "clase": class_info["display_name"],
            "señales": len(formatted_files),
            "muestras_por_señal": sample.shape[0],
            "duración_s": DURATION_SECONDS,
            "frecuencia_muestreo_Hz": SAMPLING_RATE_HZ,
            "formato_Edge_Impulse": "timestamp,ecg",
        }
    )

axes[-1].set_xlabel("Tiempo (s)")
fig.tight_layout()
plt.show()
display(pd.DataFrame(verification_rows))

## Importar en Edge Impulse

Usa los archivos de `dataset/<clase>/formateado/` para cargar los datos. Importa los CSV como datos de series temporales, usando `timestamp` como columna de tiempo (milisegundos) y `ecg` como sensor. Conserva los prefijos de clase para identificar las etiquetas. La carpeta `raw/` y `manifest.csv` son para referencia y no forman parte del lote de CSV formateados que se sube.